# 05 - Comparison with External Restoration Architectures
**Addresses comment 5.**  NAFNet, Restormer and SwinIR (compact re-implementations, see `models/baselines.py`) are trained from scratch with **the same split, augmentation, epochs, optimiser, schedule, early-stopping rule and multi-seed protocol** as the proposed model (notebook 03). `small` variants match the proposed model's parameter budget (~0.1 M); `base` variants are larger references. Parameters and FLOPs are always reported.

**State in the paper:** these are *re-implementations written for this repository* and not the official code; for a camera-ready table, add the official implementations through `BASELINE_REGISTRY` (same call signature) and report both. Hyper-parameter selection was identical for every method (none beyond the shared protocol).

In [1]:
import os, sys, json, warnings, importlib
PROJECT_ROOT = os.path.abspath("..")
if not os.path.isdir(os.path.join(PROJECT_ROOT, "notebooks")):
    PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)
# Prefer a configured data root only when it actually contains the paired folders.
def _has_paired_data(root):
    if not root:
        return False
    return ((os.path.isdir(os.path.join(root, "train", "train", "GT")) and
             os.path.isdir(os.path.join(root, "train", "train", "NoisyLR"))) or
            (os.path.isdir(os.path.join(root, "GT")) and
             os.path.isdir(os.path.join(root, "NoisyLR"))))
if not _has_paired_data(os.environ.get("SEMICON_DATA")):
    os.environ["SEMICON_DATA"] = PROJECT_ROOT
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import config as C
C = importlib.reload(C)
# Print explicit results and data roots below
from utils.experiment import load_results, load_per_image, EXTERNAL, build_model, CONFIGS
from utils.analysis import count_flops
from utils import stats as S, figures as FG
RUNS = os.path.join(PROJECT_ROOT, "results_kaggle")
DATA = PROJECT_ROOT
RUNS_DIR = os.path.join(RUNS, "runs")
TABLES_DIR = os.path.join(RUNS, "tables")
FIGS_DIR = os.path.join(RUNS, "figures")
print("RUNS =", RUNS)
print("DATA =", DATA)
if not os.path.isdir(os.path.join(DATA, "train", "train", "GT")) or not os.path.isdir(os.path.join(DATA, "train", "train", "NoisyLR")):
    raise FileNotFoundError(f"Paired training data not found under DATA={DATA}")
res = load_results(RUNS_DIR)
if res.empty: raise FileNotFoundError(f"No completed runs found under RUNS={RUNS_DIR}")
have = [c for c in EXTERNAL if c in set(res.config)]
print("External configs found:", have)
if not have:
    print("No external baselines are in results_kaggle yet.")
    print(rf'To train them into this folder, run in PowerShell: $env:SEMICON_RUNS = "{RUNS}"; python scripts/run_experiments.py --mode pilot --groups external --seeds 0,1')

RUNS = d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow\results_kaggle
DATA = d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow
External configs found: ['baseline', 'full', 'NAFNet-small', 'Restormer-small', 'SwinIR-small']


In [2]:
rows = []
for c in have:
    g = res[res.config == c]; m = build_model(CONFIGS[c])
    rows.append(dict(config=c, params=int(g.params.iloc[0]), GFLOPs_128=round(count_flops(m, (1, 1, 128, 128)), 2), n_seeds=len(g),
                     PSNR=f"{g.test_psnr.mean():.3f} +- {g.test_psnr.std(ddof=1) if len(g)>1 else float('nan'):.3f}",
                     SSIM=f"{g.test_ssim.mean():.4f} +- {g.test_ssim.std(ddof=1) if len(g)>1 else float('nan'):.4f}",
                     HP_Err=f"{g.test_hp_err.mean():.4f}", edge_corr=f"{g.test_edge_corr.mean():.4f}", epochs_run=int(g.best_epoch.max())))
ext = pd.DataFrame(rows, columns=["config", "params", "GFLOPs_128", "n_seeds", "PSNR", "SSIM", "HP_Err", "edge_corr", "epochs_run"])
print(ext.to_string(index=False) if have else "No external model results to summarize yet.")
ext.to_csv(os.path.join(TABLES_DIR, "table_external_main.csv"), index=False)

         config  params  GFLOPs_128  n_seeds            PSNR             SSIM HP_Err edge_corr  epochs_run
       baseline   86945        3.90        4 27.646 +- 0.055 0.7290 +- 0.0019 0.1074    0.8523          30
           full  116138        4.84        4 27.769 +- 0.028 0.7331 +- 0.0007 0.1057    0.8557          30
   NAFNet-small  112817        0.73        4 27.684 +- 0.029 0.7314 +- 0.0009 0.1101    0.8495          30
Restormer-small  111917        3.87        4 27.923 +- 0.014 0.7399 +- 0.0005 0.1044    0.8585          30
   SwinIR-small  100813        3.97        4 27.852 +- 0.017 0.7369 +- 0.0006 0.1052    0.8580          30


In [3]:
pi = {c: load_per_image(RUNS_DIR, c, "psnr") for c in have}
out = []
for ref in [r for r in ["full", "baseline"] if r in have]:
    t = S.compare_to_reference(pi, ref, have); t["metric"] = "psnr"; out.append(t)
pt = pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=["config", "vs", "mean_diff", "ci95_lo", "ci95_hi", "p_wilcoxon_holm", "cohens_dz", "significant_0.05", "metric"])
if not pt.empty:
    print("(mean_diff > 0 = config better than reference)")
    print(pt[["config", "vs", "mean_diff", "ci95_lo", "ci95_hi", "p_wilcoxon_holm", "cohens_dz", "significant_0.05"]].round(4).to_string(index=False))
else: print("No external paired tests available because external runs have not been trained.")
pt.to_csv(os.path.join(TABLES_DIR, "table_external_paired_tests.csv"), index=False)

(mean_diff > 0 = config better than reference)
         config       vs  mean_diff  ci95_lo  ci95_hi  p_wilcoxon_holm  cohens_dz  significant_0.05
       baseline     full    -0.1231  -0.1366  -0.1102           0.0000    -1.0034              True
   NAFNet-small     full    -0.0851  -0.1112  -0.0584           0.0000    -0.3466              True
Restormer-small     full     0.1536   0.1368   0.1714           0.0000     0.9799              True
   SwinIR-small     full     0.0829   0.0653   0.1021           0.0000     0.4961              True
           full baseline     0.1231   0.1102   0.1366           0.0000     1.0034              True
   NAFNet-small baseline     0.0380   0.0058   0.0710           0.5585     0.1269             False
Restormer-small baseline     0.2767   0.2501   0.3049           0.0000     1.1060              True
   SwinIR-small baseline     0.2060   0.1803   0.2344           0.0000     0.8320              True


In [4]:
# Honest reading rule: is the proposed model competitive at comparable size?
if "full" in have:
    small = [c for c in have if c.endswith("-small")]
    for c in small:
        r = pt[(pt.config == c) & (pt.vs == "full")]
        if len(r):
            r = r.iloc[0]; better = "BETTER than" if (r.mean_diff > 0 and r.ci95_lo > 0) else ("WORSE than" if (r.mean_diff < 0 and r.ci95_hi < 0) else "statistically INDISTINGUISHABLE from")
            print(f"{c:18s} is {better} the proposed model (diff {r.mean_diff:+.4f} dB, CI [{r.ci95_lo:+.4f}, {r.ci95_hi:+.4f}])")
    print("\nIf an external model is better or indistinguishable at equal size, the paper must say the mechanism does not provide a demonstrated advantage over the choice of backbone.")
if have:
    sm = S.summarize_configs(res, "test_psnr", have)
    FG.bars_with_error(sm, ["test_psnr"], "External comparison - test PSNR (mean +- std over seeds)", FIGS_DIR, "fig_external_psnr", ylabel="PSNR (dB)")
else:
    print("External comparison figure skipped until external runs are available.")

NAFNet-small       is WORSE than the proposed model (diff -0.0851 dB, CI [-0.1112, -0.0584])
Restormer-small    is BETTER than the proposed model (diff +0.1536 dB, CI [+0.1368, +0.1714])
SwinIR-small       is BETTER than the proposed model (diff +0.0829 dB, CI [+0.0653, +0.1021])

If an external model is better or indistinguishable at equal size, the paper must say the mechanism does not provide a demonstrated advantage over the choice of backbone.
